In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, T5EncoderModel, AutoTokenizer, AutoModelForMaskedLM
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

BASE_DIR = "/kaggle/working/polyglot_ai_detector_v2" 
MODEL_WEIGHTS = os.path.join(BASE_DIR, "polyglot_temporal_final.pt")
REPORT_CSV = os.path.join(BASE_DIR, "semeval_final_audit.csv")
PLOT_PNG = os.path.join(BASE_DIR, "semeval_confusion_matrix.png")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 256
METRIC_DIM = 7

semeval_ds = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="validation")
semeval_ds = semeval_ds.select(range(1000))

cb_tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base-mlm")
cb_model = AutoModelForMaskedLM.from_pretrained("microsoft/codebert-base-mlm").to(DEVICE)
cb_model.eval()

t5_tokenizer = RobertaTokenizer.from_pretrained("Salesforce/codet5-base", extra_special_tokens=[])
base_t5 = T5EncoderModel.from_pretrained("Salesforce/codet5-base")

class MageCodeClassifier(nn.Module):
    def __init__(self, base, metric_dim=METRIC_DIM):
        super().__init__()
        self.base = base
        h = base.config.hidden_size 
        
        self.metric_cnn = nn.Sequential(
            nn.Conv1d(in_channels=metric_dim, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) 
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(h + 64, 1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 1)
        )

    def forward(self, input_ids, attention_mask, metric_vector):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state            
        mask = attention_mask.unsqueeze(-1).float()          
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-4)
        
        metric_vector = metric_vector.transpose(1, 2)
        cnn_features = self.metric_cnn(metric_vector).squeeze(-1) 
        
        combined = torch.cat([pooled, cnn_features], dim=1)
        return self.classifier(combined) 

detector = MageCodeClassifier(base_t5).to(DEVICE)
detector.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=DEVICE))
detector.eval()

all_preds = []
all_probs = []
all_labels = semeval_ds["label"]

with torch.no_grad():
    for code_snippet in tqdm(semeval_ds["code"], desc="Inference"):
        cb_inputs = cb_tokenizer(code_snippet, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_LEN).to(DEVICE)
        
        with torch.amp.autocast("cuda"):
            cb_logits = cb_model(**cb_inputs).logits
            
        seq_len = cb_inputs["attention_mask"][0].sum().item()
        padded_metrics = torch.zeros((1, MAX_LEN, 7), dtype=torch.float32).to(DEVICE)
        
        if seq_len > 1:
            seq_logits = cb_logits[0:1, :seq_len-1, :]
            seq_labels = cb_inputs["input_ids"][0:1, 1:seq_len]
            probs = F.softmax(seq_logits, dim=-1)
            
            entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1)
            target_probs = probs.gather(2, seq_labels.unsqueeze(-1)).squeeze(-1)
            log_likelihood = torch.log(target_probs + 1e-9)
            
            sorted_indices = torch.argsort(seq_logits, dim=-1, descending=True)
            ranks = (sorted_indices == seq_labels.unsqueeze(-1)).nonzero(as_tuple=True)[2].view(1, -1) + 1
            log_rank = torch.log(ranks.float())
            
            token_metrics = torch.stack([
                log_likelihood, log_rank, entropy,
                (ranks <= 10).float(),
                ((ranks > 10) & (ranks <= 100)).float(),
                ((ranks > 100) & (ranks <= 1000)).float(),
                (ranks > 1000).float()
            ], dim=-1)
            
            padded_metrics[0, :token_metrics.size(1), :] = token_metrics[0]
        
        clean_metrics = torch.nan_to_num(padded_metrics, nan=0.0, posinf=10.0, neginf=-100.0)
        t5_inputs = t5_tokenizer(code_snippet, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_LEN).to(DEVICE)
        
        with torch.amp.autocast("cuda"):
            logits = detector(t5_inputs["input_ids"], t5_inputs["attention_mask"], clean_metrics).view(-1)
            prob = torch.sigmoid(logits).item()
            
        all_probs.append(prob)
        all_preds.append(1 if prob > 0.5 else 0)

target_names = ["Human Written (0)", "AI Generated (1)"]
print(classification_report(all_labels, all_preds, target_names=target_names, digits=4))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.ylabel('Actual Origin')
plt.xlabel('Predicted Origin')
plt.title('Detector Confusion Matrix')
plt.tight_layout()
plt.savefig(PLOT_PNG)

df = pd.DataFrame({
    "Code_Preview": [c[:100].replace('\n', ' ') for c in semeval_ds["code"]],
    "True_Label": all_labels,
    "Predicted_Label": all_preds,
    "AI_Probability": all_probs
})
df.to_csv(REPORT_CSV, index=False)

accuracy = (df["True_Label"] == df["Predicted_Label"]).mean() * 100
print(f"Final Accuracy: {accuracy:.2f}%")